# TimesNet on Server - Setup and Training

This notebook contains all commands needed to run TimesNet on the remote server.

## Check GPU Availability

In [1]:
!nvidia-smi

Tue Mar  3 12:38:24 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.230.02             Driver Version: 535.230.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 2080        Off | 00000000:02:00.0 Off |                  N/A |
| 41%   29C    P8               3W / 225W |      1MiB /  8192MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

## Check Current Directory

In [ ]:
!pwd
!ls -la

## Navigate to Project Directory

In [ ]:
%cd /home/fzf/dev/TSAD/TimesNet/Time-Series-Library

## Setup Conda Environment (First Time Only)

Run these cells only once to create the environment:

In [ ]:
# Create conda environment (only run once)
!conda create -n timesnet python=3.9 -y

In [ ]:
# Register environment as Jupyter kernel (only run once)
!conda run -n timesnet pip install ipykernel
!conda run -n timesnet python -m ipykernel install --user --name=timesnet --display-name "Python (timesnet)"

## Install Dependencies

**IMPORTANT:** After running the cells above, switch the kernel to "Python (timesnet)" using the kernel selector in the top right!

Then run these cells to install packages:

In [1]:
# Verify we're using the correct environment
import sys
print(f"Python path: {sys.executable}")
print(f"Should contain 'timesnet' in the path")

Python path: /home/fzf/.conda/envs/timesnet/bin/python
Should contain 'timesnet' in the path


In [ ]:
# Install PyTorch with CUDA support
!pip install torch torchvision torchaudio

In [ ]:
# Install other dependencies
!pip install einops reformer-pytorch local-attention sktime sympy PyWavelets patool tqdm huggingface_hub datasets pandas numpy matplotlib scikit-learn

In [ ]:
# Install additional requirements if requirements.txt exists
!pip install -r requirements.txt

## Verify GPU is Available in Python

In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
CUDA version: 12.8
GPU count: 4
Current GPU: NVIDIA GeForce RTX 2080


## Run TimesNet on MSL Dataset

Choose which GPU to use by setting CUDA_VISIBLE_DEVICES:

In [21]:
# Set which GPU to use (0, 1, 2, etc.)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2,3'  # Change this number based on nvidia-smi output

## SMD Dataset

In [40]:
!python -u run.py \
  --task_name anomaly_detection \
  --is_training 1 \
  --root_path ./dataset/SMD \
  --model_id SMD \
  --model TimesNet \
  --data SMD \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 64 \
  --d_ff 64 \
  --e_layers 2 \
  --enc_in 38 \
  --c_out 38 \
  --top_k 5 \
  --anomaly_ratio 0.5 \
  --batch_size 128 \
  --train_epochs 10

Using GPU
Args in experiment:
Basic Config
  Task Name:          anomaly_detection   Is Training:        1                   
  Model ID:           SMD                 Model:              TimesNet            

Data Loader
  Data:               SMD                 Root Path:          ./dataset/SMD       
  Data Path:          ETTh1.csv           Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Anomaly Detection Task
  Anomaly Ratio:      0.5                 

Model Parameters
  Top k:              5                   Num Kernels:        6                   
  Enc In:             38                  Dec In:             7                   
  C Out:              38                  d model:            64                  
  n heads:            8                   e layers:           2                   
  d layers:           1                   d FF:               64     

In [ ]:
# ── Count MACs for TimesNet on ALL datasets (best hyperparams) ─────────
import sys, argparse
import torch
import torch.nn as nn

sys.path.insert(0, './')
from models.TimesNet import Model as TimesNet
from fvcore.nn import FlopCountAnalysis
from thop import profile

class _Wrap(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.inner = m
    def forward(self, x):
        return self.inner(x, None, None, None)

# Best hyperparams per dataset (from result_anomaly_detection.txt)
datasets = {
    # SMD:  F1=0.8459  dm64 df64 el2
    'SMD':   dict(seq_len=100,  enc_in=38, c_out=38, d_model=64,  d_ff=64,  e_layers=2, top_k=5),
    # MSL:  F1=0.8180  dm8 df16 el1
    'MSL':   dict(seq_len=100,  enc_in=55, c_out=55, d_model=8,   d_ff=16,  e_layers=1, top_k=3),
    # SMAP: F1=0.6944  dm128 df128 el3
    'SMAP':  dict(seq_len=100,  enc_in=25, c_out=25, d_model=128, d_ff=128, e_layers=3, top_k=3),
    # SWaT: F1=0.9262  dm8 df8 el3
    'SWaT':  dict(seq_len=100,  enc_in=51, c_out=51, d_model=8,   d_ff=8,   e_layers=3, top_k=3),
    # PSM:  F1=0.9738  dm64 df64 el2
    'PSM':   dict(seq_len=100,  enc_in=25, c_out=25, d_model=64,  d_ff=64,  e_layers=2, top_k=3),
    # GECCO: F1=0.4597  sl800 dm32 df32 el3
    'GECCO': dict(seq_len=800,  enc_in=9,  c_out=9,  d_model=32,  d_ff=32,  e_layers=3, top_k=1),
}

print(f"{'Dataset':<8} {'fvcore MACs':>14} {'thop MACs':>14} {'Diff%':>10} {'Params':>10}")
print("-" * 63)

for name, hp in datasets.items():
    cfg = argparse.Namespace(
        task_name='anomaly_detection',
        label_len=48, pred_len=0,
        num_kernels=6, embed='timeF', freq='h', dropout=0.1,
        **hp,
    )
    model = TimesNet(cfg).cpu().eval()
    wrapped = _Wrap(model)
    torch.manual_seed(42) # To get reproducible numbers across runs
    dummy = torch.randn(1, cfg.seq_len, cfg.enc_in)

    with torch.no_grad():
        # fvcore
        fa = FlopCountAnalysis(wrapped, dummy)
        fa.unsupported_ops_warnings(False)
        fa.uncalled_modules_warnings(False)
        fvcore_macs = fa.total()

        # thop
        thop_macs, thop_params = profile(wrapped, inputs=(dummy,), verbose=False)

    diff_pct = abs(fvcore_macs - thop_macs) / max(fvcore_macs, 1) * 100
    print(f"{name:<8} {fvcore_macs:>14,} {thop_macs:>14,.0f} {diff_pct:>9.4f}% {thop_params:>10,.0f}")

Dataset     fvcore MACs      thop MACs      Diff%     Params
---------------------------------------------------------------
SMD       2,428,293,632  2,428,280,832    0.0005%  4,697,254
MSL          22,876,960     22,876,160    0.0035%     75,191
SMAP      8,773,334,528  8,773,296,128    0.0004% 28,132,633
SWaT         33,268,832     33,266,432    0.0072%    111,811
PSM       1,462,681,088  1,462,668,288    0.0009%  4,693,913
GECCO     1,407,052,800  1,406,976,000    0.0055%  1,759,561


In the results, SMD has ~4.7M params because it uses d_model=64, d_ff=64, e_layers=2 (larger hidden dims), while SWaT has only ~112K params because it uses d_model=8, d_ff=8 (much smaller). The parameter count determines how much memory the model needs and is independent of input sequence length — unlike MACs, which scale with seq_len.

## MSL dataset

In [ ]:
!CUDA_VISIBLE_DEVICES=2 python -u run.py \
  --task_name anomaly_detection \
  --is_training 1 \
  --root_path ./dataset/MSL \
  --model_id MSL \
  --model TimesNet \
  --data MSL \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 8 \
  --d_ff 16 \
  --e_layers 1 \
  --enc_in 55 \
  --c_out 55 \
  --top_k 3 \
  --anomaly_ratio 1 \
  --batch_size 128 \
  --train_epochs 1

## SMAP Dataset

In [10]:
!CUDA_VISIBLE_DEVICES=3 python -u run.py \
  --task_name anomaly_detection \
  --is_training 1 \
  --root_path ./dataset/SMAP \
  --model_id SMAP \
  --model TimesNet \
  --data SMAP \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 128 \
  --d_ff 128 \
  --e_layers 3 \
  --enc_in 25 \
  --c_out 25 \
  --top_k 3 \
  --anomaly_ratio 1 \
  --batch_size 128 \
  --train_epochs 3

Using GPU
Args in experiment:
Basic Config
  Task Name:          anomaly_detection   Is Training:        1                   
  Model ID:           SMAP                Model:              TimesNet            

Data Loader
  Data:               SMAP                Root Path:          ./dataset/SMAP      
  Data Path:          ETTh1.csv           Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Anomaly Detection Task
  Anomaly Ratio:      1.0                 

Model Parameters
  Top k:              3                   Num Kernels:        6                   
  Enc In:             25                  Dec In:             7                   
  C Out:              25                  d model:            128                 
  n heads:            8                   e layers:           3                   
  d layers:           1                   d FF:               128    

## SWaT dataset

In [ ]:
!CUDA_VISIBLE_DEVICES=2 python -u run.py \
  --task_name anomaly_detection \
  --is_training 1 \
  --root_path ./dataset/SWaT \
  --model_id SWAT \
  --model TimesNet \
  --data SWAT \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 8 \
  --d_ff 8 \
  --e_layers 3 \
  --enc_in 51 \
  --c_out 51 \
  --top_k 3 \
  --anomaly_ratio 1 \
  --batch_size 128 \
  --train_epochs 3


## PSM dataset

In [ ]:
!CUDA_VISIBLE_DEVICES=2 python -u run.py \
  --task_name anomaly_detection \
  --is_training 1 \
  --root_path ./dataset/PSM \
  --model_id PSM \
  --model TimesNet \
  --data PSM \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 64 \
  --d_ff 64 \
  --e_layers 2 \
  --enc_in 25 \
  --c_out 25 \
  --top_k 3 \
  --anomaly_ratio 1 \
  --batch_size 128 \
  --train_epochs 3

## Prepare GECCO Dataset

In [11]:
!python scripts/prepare_gecco.py

Reading /home/fzf/dev/TSAD/datasets/GECCO/gecco2018_water_quality.csv...
Feature columns (9): ['Tp', 'Cl', 'pH', 'Redox', 'Leit', 'Trueb', 'Cl_2', 'Fm', 'Fm_2']
Label column: EVENT
Total samples: 139566, Features: 9
Anomaly ratio: 1.24%
Train samples: 83739, Test samples: 55827
Test anomaly ratio: 0.45%
Saved train.npy, test.npy, test_label.npy to /home/fzf/dev/TSAD/TimesNet/Time-Series-Library/dataset/GECCO

Done! Use --enc_in 9 --c_out 9 when running TimesNet.


## Run TimesNet on GECCO Dataset

In [36]:
# Run TimesNet training on GECCO dataset
!python -u run.py \
  --task_name anomaly_detection \
  --is_training 1 \
  --root_path ./dataset/GECCO \
  --model_id GECCO \
  --model TimesNet \
  --data GECCO \
  --features M \
  --seq_len 2000 \
  --pred_len 0 \
  --d_model 32 \
  --d_ff 32 \
  --e_layers 3 \
  --enc_in 9 \
  --c_out 9 \
  --top_k 1 \
  --anomaly_ratio 0.01 \
  --batch_size 128 \
  --train_epochs 3

Using GPU
Args in experiment:
Basic Config
  Task Name:          anomaly_detection   Is Training:        1                   
  Model ID:           GECCO               Model:              TimesNet            

Data Loader
  Data:               GECCO               Root Path:          ./dataset/GECCO     
  Data Path:          ETTh1.csv           Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Anomaly Detection Task
  Anomaly Ratio:      0.01                

Model Parameters
  Top k:              1                   Num Kernels:        6                   
  Enc In:             9                   Dec In:             7                   
  C Out:              9                   d model:            32                  
  n heads:            8                   e layers:           3                   
  d layers:           1                   d FF:               32     

In [15]:
import numpy as np
labels = np.load('./dataset/GECCO/test_label.npy')
print(f"Anomaly ratio: {labels.mean() * 100:.2f}%")
print(f"Total anomaly points: {labels.sum()}")
print(f"Total points: {len(labels)}")

Anomaly ratio: 0.45%
Total anomaly points: 251
Total points: 55827


### Neural Network Intelligence (NNI) toolkit

In [10]:
# In your notebook
!nnictl create --config nni_config.yml --port 8081

/home/fzf/.conda/envs/timesnet/lib/python3.9/site-packages/nni/tools/nnictl/nnictl.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-03 15:18:15] Creating experiment, Experiment ID: yjv2xmzo
[2026-03-03 15:18:15] Starting web server...
[2026-03-03 15:18:16] Setting up...
[2026-03-03 15:18:16] Web portal URLs: http://127.0.0.1:8081 http://10.28.27.64:8081 http://172.17.0.1:8081
[2026-03-03 15:18:16] To stop experiment run "nnictl stop yjv2xmzo" or "nnictl stop --all"
[2026-03-03 15:18:16] Reference: https://nni.readthedocs.io/en/stable/reference/nnictl.html


In [9]:
!nnictl stop --all

/home/fzf/.conda/envs/timesnet/lib/python3.9/site-packages/nni/tools/nnictl/nnictl.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
INFO:  Stopping experiment 2a8og5ut
INFO:  Stop experiment success.


In [6]:
!kill -9 3457115

### Step 1: Save Reconstruction Errors

We need to modify the test function to save the raw errors before thresholding.
This monkey-patches the experiment at runtime without editing the source files.

In [2]:
import sys
import os
import numpy as np
import torch
import torch.nn as nn

# Add project to path
sys.path.insert(0, './')

from exp.exp_anomaly_detection import Exp_Anomaly_Detection
from utils.tools import adjust_learning_rate, adjustment

# ── Build args to match your GECCO training run ──────────────────────
import argparse
args = argparse.Namespace(
    task_name='anomaly_detection',
    is_training=0,
    model_id='GECCO',
    model='TimesNet',
    data='GECCO',
    root_path='./dataset/GECCO',
    data_path='ETTh1.csv',
    features='M',
    target='OT',
    freq='h',
    checkpoints='./checkpoints/',
    seq_len=100,
    label_len=48,
    pred_len=0,
    enc_in=9,
    dec_in=7,
    c_out=9,
    d_model=32,
    n_heads=8,
    e_layers=3,
    d_layers=1,
    d_ff=32,
    moving_avg=25,
    factor=1,
    distil=True,
    dropout=0.1,
    embed='timeF',
    activation='gelu',
    top_k=3,
    num_kernels=6,
    anomaly_ratio=0.01,
    batch_size=128,
    train_epochs=10,
    patience=3,
    learning_rate=0.0001,
    des='test',
    loss='MSE',
    lradj='type1',
    use_amp=False,
    num_workers=10,
    itr=1,
    use_gpu=True,
    gpu=0,
    gpu_type='cuda',
    use_multi_gpu=False,
    devices='0,1,2,3',
    p_hidden_dims=[128, 128],
    p_hidden_layers=2,
    expand=2,
    d_conv=4,
    output_attention=False,
)

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# ── Build experiment and load checkpoint ─────────────────────────────
exp = Exp_Anomaly_Detection(args)

# Build the setting string to locate the checkpoint
setting = (
    f'anomaly_detection_{args.model_id}_{args.model}_{args.model_id}_'
    f'ftM_sl{args.seq_len}_ll{args.label_len}_pl{args.pred_len}_'
    f'dm{args.d_model}_nh{args.n_heads}_el{args.e_layers}_dl{args.d_layers}_'
    f'df{args.d_ff}_expand{args.expand}_dc{args.d_conv}_fc{args.factor}_'
    f'eb{args.embed}_dt{args.distil}_{args.des}_0'
)
print('Loading checkpoint from:', setting)

checkpoint_path = os.path.join(args.checkpoints, setting, 'checkpoint.pth')
exp.model.load_state_dict(torch.load(checkpoint_path, map_location='cpu'))
exp.model.eval()

# ── Compute reconstruction errors on train set ───────────────────────
train_data, train_loader = exp._get_data(flag='train')
test_data,  test_loader  = exp._get_data(flag='test')

anomaly_criterion = nn.MSELoss(reduce=False)

train_energy = []
with torch.no_grad():
    for batch_x, _ in train_loader:
        batch_x = batch_x.float().to(exp.device)
        outputs = exp.model(batch_x, None, None, None)
        score = torch.mean(anomaly_criterion(batch_x, outputs), dim=-1)
        train_energy.append(score.detach().cpu().numpy())
train_energy = np.concatenate(train_energy).reshape(-1)

# ── Compute reconstruction errors on test set ────────────────────────
test_energy = []
test_labels = []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.float().to(exp.device)
        outputs = exp.model(batch_x, None, None, None)
        score = torch.mean(anomaly_criterion(batch_x, outputs), dim=-1)
        test_energy.append(score.detach().cpu().numpy())
        test_labels.append(batch_y.numpy())
test_energy = np.concatenate(test_energy).reshape(-1)
test_labels = np.concatenate(test_labels).reshape(-1).astype(int)

# Save for reuse
np.save('gecco_train_energy.npy', train_energy)
np.save('gecco_test_energy.npy',  test_energy)
np.save('gecco_test_labels.npy',  test_labels)

print(f'Train energy shape: {train_energy.shape}')
print(f'Test  energy shape: {test_energy.shape}')
print(f'Test  labels shape: {test_labels.shape}')
print(f'Anomaly points in test: {test_labels.sum()} / {len(test_labels)}')


Use GPU: cuda:0
Loading checkpoint from: anomaly_detection_GECCO_TimesNet_GECCO_ftM_sl100_ll48_pl0_dm32_nh8_el3_dl1_df32_expand2_dc4_fc1_ebtimeF_dtTrue_test_0
test: (55827, 9)
train: (83739, 9)
train 83640
test: (55827, 9)
train: (83739, 9)
test 55728
Train energy shape: (8364000,)
Test  energy shape: (5572800,)
Test  labels shape: (5572800,)
Anomaly points in test: 25100 / 5572800


### Step 2: Plot Reconstruction Error Distributions

This shows whether anomaly points produce clearly higher errors than normal points.
If the distributions heavily overlap, no threshold will give good F1.

In [3]:
import numpy as np
import matplotlib.pyplot as plt

train_energy = np.load('gecco_train_energy.npy')
test_energy  = np.load('gecco_test_energy.npy')
test_labels  = np.load('gecco_test_labels.npy')

normal_errors  = test_energy[test_labels == 0]
anomaly_errors = test_energy[test_labels == 1]

print(f'Normal  errors — mean: {normal_errors.mean():.4f},  std: {normal_errors.std():.4f}')
print(f'Anomaly errors — mean: {anomaly_errors.mean():.4f},  std: {anomaly_errors.std():.4f}')
print(f'Separation ratio (anomaly_mean / normal_mean): {anomaly_errors.mean() / normal_errors.mean():.2f}x')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ── Left: overlapping histogram ──────────────────────────────────────
axes[0].hist(normal_errors,  bins=100, alpha=0.6, label=f'Normal  (n={len(normal_errors):,})', density=True, color='steelblue')
axes[0].hist(anomaly_errors, bins=30,  alpha=0.7, label=f'Anomaly (n={len(anomaly_errors)})',   density=True, color='tomato')
axes[0].set_xlabel('Reconstruction Error')
axes[0].set_ylabel('Density')
axes[0].set_title('GECCO: Error Distribution (Normal vs Anomaly)')
axes[0].legend()

# ── Right: log-scale to see tail separation ──────────────────────────
axes[1].hist(normal_errors,  bins=100, alpha=0.6, label='Normal',  density=True, color='steelblue')
axes[1].hist(anomaly_errors, bins=30,  alpha=0.7, label='Anomaly', density=True, color='tomato')
axes[1].set_yscale('log')
axes[1].set_xlabel('Reconstruction Error')
axes[1].set_ylabel('Density (log scale)')
axes[1].set_title('GECCO: Error Distribution (Log Scale)')
axes[1].legend()

plt.tight_layout()
plt.savefig('gecco_error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gecco_error_distribution.png')


Normal  errors — mean: 0.0055,  std: 0.1357
Anomaly errors — mean: 0.7340,  std: 2.6125
Separation ratio (anomaly_mean / normal_mean): 133.49x
Saved: gecco_error_distribution.png


### Step 3: F1 Score vs Anomaly Ratio Sweep

Sweeps all threshold values to find the best possible F1,
and shows the precision-recall tradeoff curve.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
from utils.tools import adjustment

train_energy = np.load('gecco_train_energy.npy')
test_energy  = np.load('gecco_test_energy.npy')
test_labels  = np.load('gecco_test_labels.npy')

combined_energy = np.concatenate([train_energy, test_energy])

# Sweep anomaly_ratio from 0.01% to 20%
ratios     = np.logspace(-2, np.log10(20), 60)  # log-spaced
results    = []

for ratio in ratios:
    threshold = np.percentile(combined_energy, 100 - ratio)
    pred = (test_energy > threshold).astype(int)
    gt   = test_labels.copy()
    gt, pred = adjustment(gt, pred)   # point-adjustment (same as paper)
    p, r, f, _ = precision_recall_fscore_support(gt, pred, average='binary', zero_division=0)
    results.append({'ratio': ratio, 'precision': p, 'recall': r, 'f1': f})

ratios_arr = [r['ratio']     for r in results]
f1_arr     = [r['f1']        for r in results]
prec_arr   = [r['precision'] for r in results]
rec_arr    = [r['recall']    for r in results]

best_idx   = int(np.argmax(f1_arr))
best_ratio = ratios_arr[best_idx]
best_f1    = f1_arr[best_idx]
best_prec  = prec_arr[best_idx]
best_rec   = rec_arr[best_idx]

print(f'Best anomaly_ratio : {best_ratio:.4f}%')
print(f'Best F1            : {best_f1*100:.2f}%')
print(f'Precision at best  : {best_prec*100:.2f}%')
print(f'Recall    at best  : {best_rec*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ── Left: F1 vs anomaly_ratio ────────────────────────────────────────
axes[0].semilogx(ratios_arr, f1_arr,   label='F1',        color='green')
axes[0].semilogx(ratios_arr, prec_arr, label='Precision', color='steelblue', linestyle='--')
axes[0].semilogx(ratios_arr, rec_arr,  label='Recall',    color='tomato',    linestyle='--')
axes[0].axvline(best_ratio, color='gray', linestyle=':', label=f'Best ratio={best_ratio:.3f}%')
axes[0].set_xlabel('anomaly_ratio (%, log scale)')
axes[0].set_ylabel('Score')
axes[0].set_title('GECCO: Metrics vs Threshold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Right: Precision-Recall curve ───────────────────────────────────
axes[1].plot(rec_arr, prec_arr, color='purple', marker='o', markersize=3)
axes[1].scatter([best_rec], [best_prec], color='red', zorder=5, s=80, label=f'Best F1={best_f1*100:.1f}%')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('GECCO: Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gecco_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gecco_threshold_sweep.png')


Best anomaly_ratio : 1.9676%
Best F1            : 39.09%
Precision at best  : 24.89%
Recall    at best  : 91.02%
Saved: gecco_threshold_sweep.png


### Step 4: Visualise Raw Data Around Anomalies

Plots the raw sensor signals around the actual anomaly events
to understand what the anomalies look like.

In [5]:
import numpy as np
import matplotlib.pyplot as plt

test_data   = np.load('./dataset/GECCO/test.npy')
test_labels = np.load('gecco_test_labels.npy')
test_energy = np.load('gecco_test_energy.npy')

feature_names = ['Tp', 'Cl', 'pH', 'Redox', 'Leit', 'Trueb', 'Cl_2', 'Fm', 'Fm_2']

anomaly_idx = np.where(test_labels == 1)[0]
print(f'Anomaly indices: {anomaly_idx}')

# Find contiguous anomaly segments
gaps = np.where(np.diff(anomaly_idx) > 1)[0]
segments = np.split(anomaly_idx, gaps + 1)
print(f'Number of anomaly segments: {len(segments)}')
for i, seg in enumerate(segments):
    print(f'  Segment {i+1}: indices {seg[0]}–{seg[-1]}, length {len(seg)}')

# Plot window around first anomaly segment
seg = segments[0]
window = 200  # time points to show before/after
start  = max(0, seg[0] - window)
end    = min(len(test_data), seg[-1] + window)

fig, axes = plt.subplots(3, 3, figsize=(15, 8), sharex=True)
axes = axes.flatten()

for i, name in enumerate(feature_names):
    axes[i].plot(range(start, end), test_data[start:end, i], color='steelblue', linewidth=0.8)
    # Shade anomaly region
    for s in segments:
        s_clip = s[(s >= start) & (s < end)]
        if len(s_clip) > 0:
            axes[i].axvspan(s_clip[0], s_clip[-1], alpha=0.3, color='tomato')
    axes[i].set_title(name)
    axes[i].grid(True, alpha=0.3)

fig.suptitle('GECCO: Raw sensor signals around anomaly (red = anomaly)', fontsize=13)
plt.tight_layout()
plt.savefig('gecco_raw_anomaly.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gecco_raw_anomaly.png')


Anomaly indices: [ 772999  773098  773099 ... 4860900 4860901 4861000]
Number of anomaly segments: 944
  Segment 1: indices 772999–772999, length 1
  Segment 2: indices 773098–773099, length 2
  Segment 3: indices 773197–773199, length 3
  Segment 4: indices 773296–773299, length 4
  Segment 5: indices 773395–773399, length 5
  Segment 6: indices 773494–773499, length 6
  Segment 7: indices 773593–773599, length 7
  Segment 8: indices 773692–773699, length 8
  Segment 9: indices 773791–773799, length 9
  Segment 10: indices 773890–773899, length 10
  Segment 11: indices 773989–773999, length 11
  Segment 12: indices 774088–774099, length 12
  Segment 13: indices 774187–774199, length 13
  Segment 14: indices 774286–774299, length 14
  Segment 15: indices 774385–774399, length 15
  Segment 16: indices 774484–774499, length 16
  Segment 17: indices 774583–774599, length 17
  Segment 18: indices 774682–774699, length 18
  Segment 19: indices 774781–774799, length 19
  Segment 20: indices 

## Useful Commands for Monitoring

In [ ]:
# Check what's in the current directory
!ls -lh

In [ ]:
# Check GPU memory usage
!nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total --format=csv

In [ ]:
# List saved checkpoints/results
!ls -lh checkpoints/